In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import math
import os
import shutil

In [ ]:
# Plot the rmse imputation longest interval based on a file with rmse results
def plot_rmse_imputation(rmse_results_file_path='rmse_results.json'):
    with open(rmse_results_file_path, 'r') as f:
        json_data = json.load(f)

    # Function to parse the protocol and link from the file name
    def parse_file_name(file_name):
        parts = file_name.split()
        protocol = parts[1]  # Second word is the protocol
        try:
            link_index = parts.index('data') + 1
            link = parts[link_index]
        except ValueError:
            link = 'Unknown'
        return protocol, link

    # Function to load and process the data
    def load_data(json_data):
        data_list = []
        for file_name, techniques in json_data.items():
            protocol, link = parse_file_name(file_name)
            for imputation, metrics in techniques.items():
                throughput = metrics['Throughput']
                data_list.append({
                    'File': file_name,
                    'Protocol': protocol,
                    'Link': link,
                    'Imputation': imputation,
                    'Throughput': throughput
                })
        return data_list
    
    def plot_data(data_list):
        df = pd.DataFrame(data_list)
        # Create a shorter label for plotting
        df['Label'] = df['Protocol'] + ' ' + df['Link']

        # Pivot the DataFrame to have 'Label' as index and 'Imputation' as columns
        pivot_df = df.pivot(index='Label', columns='Imputation', values='Throughput')

        # Plot the data
        pivot_df.plot(kind='bar', figsize=(14, 8))
        plt.xlabel('Protocol and Link')
        plt.ylabel('Throughput')
        plt.title('Throughput by Imputation Technique, Protocol, and Link')
        plt.legend(title='Imputation Technique', bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout()
        plt.show()

    data_list = load_data(json_data)
    
    plot_data(data_list)

def order_files_by_svd(rmse_results_file_path='rmse_results.json'):
    file_names = []
    # Load the RMSE data from the JSON file
    with open(rmse_results_file_path, 'r') as f:
        rmse_results = json.load(f)

    # Initialize a list to store files with their scores
    files_scores = []

    # Loop over each file in the RMSE results
    for filename, techniques in rmse_results.items():
        # Check if 'svd' is among the techniques for this file
        if 'svd' not in techniques:
            continue  # Skip if 'svd' data is not available

        # Get the RMSE value for 'svd' (assuming 'Throughput' is the metric)
        svd_rmse = techniques['svd'].get('Throughput')
        if svd_rmse == 0:
            continue # Skip if 'svd' RMSE is 0
        
        if svd_rmse is None or math.isnan(svd_rmse):
            continue  # Skip if 'Throughput' metric is missing or is NaN

        # Collect RMSEs of other techniques
        other_rmses = []
        for tech_name, rmse_dict in techniques.items():
            if tech_name == 'svd':
                continue  # Skip 'svd' itself

            other_rmse = rmse_dict.get('Throughput')
            if other_rmse is None or math.isnan(other_rmse):
                continue  # Skip if other RMSE is NaN

            other_rmses.append(other_rmse)

        # If there are no other techniques with valid RMSEs, skip this file
        if not other_rmses:
            continue

        # Compute the average RMSE of other techniques
        avg_other_rmse = sum(other_rmses) / len(other_rmses)

        # Compute the percentage difference
        percentage_difference = ((avg_other_rmse - svd_rmse) / avg_other_rmse) * 100

        # Add to the list
        files_scores.append((filename, percentage_difference))

    # Sort the list by percentage difference in ascending order (lowest percentage first)
    files_scores.sort(key=lambda x: x[1], reverse=True)

    # Print the sorted list of files with their percentage differences
    print("Files ordered by percentage difference between 'svd' RMSE and average RMSE of other techniques:")
    for filename, percentage_difference in files_scores:
        print(f"{filename}: Percentage Difference = {percentage_difference:.2f}%")
        file_names.append(filename)
    
    return file_names


# ex choosen_files = [
#     'treated bbr esmond data es-se 07-03-2023_longest_interval.csv',
#     'treated bbr esmond data pb-ap 07-08-2023_longest_interval.csv',
#     'treated bbr esmond data ap-se 07-03-2023_longest_interval.csv',
#     'treated bbr esmond data rj-go 07-08-2023_longest_interval.csv'
# ]
def plot_chosen_files(choosen_files, rmse_result_path):
    with open(rmse_result_path, 'r') as f:
        json_data = json.load(f)

    # Function to parse the protocol and link from the file name
    def parse_file_name(file_name):
        parts = file_name.split()
        protocol = parts[1]  # Second word is the protocol
        try:
            link_index = parts.index('data') + 1
            link = parts[link_index]
        except ValueError:
            link = 'Unknown'
        return protocol, link

    # Function to load and process the data
    def load_data(json_data, choosen_files):
        data_list = []
        for file_name, techniques in json_data.items():
            if file_name in choosen_files:
                protocol, link = parse_file_name(file_name)
                for imputation, metrics in techniques.items():
                    throughput = metrics.get('Throughput')
                    if throughput is not None:
                        data_list.append({
                            'File': file_name,
                            'Protocol': protocol,
                            'Link': link,
                            'Imputation': imputation,
                            'Throughput': throughput
                        })
        return data_list

    # Function to plot the data
    def plot_data(data_list):
        df = pd.DataFrame(data_list)
        # Create a shorter label for plotting
        df['Label'] = df['Protocol'] + ' ' + df['Link']
        
        # Pivot the DataFrame to have 'Label' as index and 'Imputation' as columns
        pivot_df = df.pivot(index='Label', columns='Imputation', values='Throughput')
        
        # Plot the data
        pivot_df.plot(kind='bar', figsize=(14, 8))
        plt.xlabel('Protocol and Link')
        plt.ylabel('Throughput')
        plt.title('Throughput by Imputation Technique, Protocol, and Link')
        plt.legend(title='Imputation Technique', bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout()
        plt.show()

    # Load and process the data
    data_list = load_data(json_data, choosen_files)

    # Plot the data
    plot_data(data_list)


def move_choosen_files(choosen_files, datasets_choosen_path, destination_folder):
    # Create the destination folder if it doesn't exist
    os.makedirs(destination_folder, exist_ok=True)

    # Dictionary to keep track of found files
    found_files = {}

    # Walk through the directory tree to find the files
    for dirpath, dirnames, filenames in os.walk(datasets_choosen_path):
        print(filenames)
        # print(datasets_choosen_path)
        for filename in filenames:
            print(filename)
            file_name_longest = filename.replace('.csv', '_longest_interval.csv')
            #print(file_name_longest)
            if file_name_longest in choosen_files:
                source_file = os.path.join(dirpath, filename)
                print(source_file)
                destination_file = os.path.join(destination_folder, file_name_longest)
                print(destination_file)
                shutil.copy2(source_file, destination_file)
                print(f"Copied: {filename} from {dirpath}")
                # Keep track of found files
                found_files[filename] = True

    # # Report files not found
    # for filename in choosen_files:
    #     if filename not in found_files:
    #         print(f"File not found: {filename}")

def create_rmse(diretorio1, diretorio2, tecnicas, rmse_results_saving_path='rmse_results.json'):
    rmse_results = {}

    # Loop over each file in the first directory
    for arquivo in os.listdir(diretorio1):
        path = os.path.join(diretorio1, arquivo)
        df = pd.read_csv(path)
        
        # Initialize a dictionary for this file
        rmse_results[arquivo] = {}
        
        # Loop over each technique
        for tecnica in tecnicas:
            path2 = os.path.join(diretorio2, tecnica, arquivo)
            
            if not os.path.exists(path2):
                continue
            
            df2 = pd.read_csv(path2)
            
            # Ensure both dataframes have the same columns
            common_columns = df.columns.intersection(df2.columns)
            df = df[common_columns]
            df2 = df2[common_columns]
            
            # Compute RMSE between df and df2 for numeric columns
            numeric_cols = df.select_dtypes(include=[np.number]).columns
            diff = df[numeric_cols] - df2[numeric_cols]
            mse = (diff ** 2).mean()
            rmse = np.sqrt(mse)
            
            # Convert RMSE Series to a dictionary for JSON serialization
            rmse_dict = rmse.to_dict()
            
            # Store the RMSE for this technique
            rmse_results[arquivo][tecnica] = rmse_dict

    # Save the RMSE results to a JSON file
    with open(rmse_results_saving_path, 'w') as f:
        json.dump(rmse_results, f, indent=4)



In [ ]:
diretorio1 = '../datasets/treated longest interval with failures/'
diretorio2 = '../datasets/imputed-treated-longest-interval-with-failures/'

tecnicas = ['knn', 'media-movel', 'mediana-movel', 'svd', 'interpolacao-linear']

results_path = '../results/rmse_longest_interval_results.json'

In [ ]:
create_rmse(diretorio1, diretorio2, tecnicas, results_path)

In [ ]:
ordered_files = order_files_by_svd(results_path)

In [ ]:
with open('../results/ordered_longest_interval_files.json', 'w') as f:
        json.dump(ordered_files, f, indent=4)

In [ ]:
choosen_files = ordered_files[:20]

In [ ]:
choosen_files

In [ ]:
with open('../results/20_choosen_files.json', 'w') as f:
        json.dump(choosen_files, f, indent=4)

In [ ]:
with open('../results/20_choosen_files.json', 'r') as f:
        choosen_files_json = json.load(f)

In [ ]:
plot_chosen_files(choosen_files_json, results_path)

In [ ]:
source_choosen = '../datasets/lowest failures treated/'
destination_choosen = '../datasets/choosen-best-svd/'

In [ ]:
move_choosen_files(choosen_files_json, source_choosen, destination_choosen)